<a href="https://colab.research.google.com/github/cemgurbey/plastic-water-segmentation/blob/main/notebooks/plastic_water_segmentation.ipynb" target="_blank">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Plastic Residue Detection on Water Bodies
**Diffusion-synthesized data + SAM 2 masks + YOLO segmentation (PyTorch)**

End-to-end pipeline: generate a synthetic training set, train an instance-segmentation
model, and run inference with a water-contamination metric.

## 1. Setup
Clone the repo and install the package. On Colab, use a GPU runtime
(**Runtime > Change runtime type > GPU**) for the diffusion and training steps.

In [ ]:
!test -d plastic-water-segmentation || git clone -q https://github.com/cemgurbey/plastic-water-segmentation.git
%cd plastic-water-segmentation
# Regular (non-editable) install: an editable install (-e) writes a .pth
# file the running Colab kernel only reads at startup, so the import in
# the next cell would fail with ModuleNotFoundError until a restart.
!pip install -q .


## 2. Configuration
All knobs live in `SynthConfig` / `TrainConfig` dataclasses — no scattered globals.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from plastic_water_seg import CATEGORIES, SynthConfig, TrainConfig
from plastic_water_seg.data import (
    DatasetSynthesizer,
    DiffusionInpainter,
    ProceduralPainter,
    generate_water_backgrounds,
    label_size_stats,
    save_label_grid,
)
from plastic_water_seg.eval import (
    contamination_index,
    evaluate_dir,
    make_yolo_eval_dataset,
    map_eval,
    predict,
    visualize,
)
from plastic_water_seg.training import train_yolo

print("Categories:", [c.name for c in CATEGORIES])

# Defaults (see config.py) now match the real photos: objects are sampled
# small (0.05%-3% of the frame, log-uniform; real median ~0.2%), regions
# never overlap, and each object is inpainted on a crop and pasted back.
synth_cfg = SynthConfig(
    image_size=(1024, 1024),
    train_count=200,  # diffusion run: 200 first, 500+ for production
    val_count=40,
    min_objects=1,
    max_objects=6,
    clip_verify=True,  # drop inpaints that CLIP says are not the class
    seed=42,
)
train_cfg = TrainConfig(model="yolo11m-seg.pt", epochs=50, imgsz=1024, batch=8)

# Production diffusion run (needs a GPU runtime + a few hours) AND real
# water photos in data/backgrounds/ (not the test images).
USE_DIFFUSION = True


## 3. Water backgrounds
**Use real water photos** (rivers, canals, lakes, sea; glare, algae, reeds,
murky and clear water) in `data/backgrounds/` — 200+ is a good target, and
never the 40 test photos. The previous diffusion run inpainted onto 6
procedural sine-wave textures, which Stable Diffusion handles poorly and
which teach the model nothing about real water.

The procedural textures below are only generated for the CPU smoke test
(`USE_DIFFUSION = False`).


In [ ]:
bg_dir = Path("data/backgrounds")
bg_dir.mkdir(parents=True, exist_ok=True)
real_bgs = [p for p in bg_dir.iterdir()
            if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
            and not p.name.startswith("water_surface_")]

if USE_DIFFUSION and len(real_bgs) < 20:
    raise RuntimeError(
        f"Found {len(real_bgs)} real backgrounds in {bg_dir}. Upload real water "
        "photos there (aim for 200+) before the diffusion run."
    )
if not real_bgs:
    paths = generate_water_backgrounds(bg_dir, count=6, size=(1024, 1024), seed=0)
else:
    paths = real_bgs

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, p in zip(axes, paths[:3]):
    ax.imshow(Image.open(p)); ax.axis("off"); ax.set_title(p.name)
plt.tight_layout(); plt.show()


## 4. Synthesize the dataset
**Quickstart (CPU-friendly):** `ProceduralPainter` draws stylized debris directly —
great for smoke-testing the pipeline in seconds.

**Production (GPU):** swap in `DiffusionInpainter`, which inpaints photorealistic
debris with Stable Diffusion and refines each mask with SAM 2.

In [ ]:
if USE_DIFFUSION:
    # Production: diffusion inpainting (needs GPU + ~4GB model download).
    painter = DiffusionInpainter(
        model_id=synth_cfg.diffusion_model,
        num_inference_steps=synth_cfg.num_inference_steps,
        guidance_scale=synth_cfg.guidance_scale,
        strength=synth_cfg.strength,
        resolution=synth_cfg.inpaint_resolution,
        context=synth_cfg.inpaint_context,
    )
else:
    # Quickstart: procedural painter (runs anywhere).
    painter = ProceduralPainter()

synth = DatasetSynthesizer(
    backgrounds_dir=bg_dir,
    output_dir=Path("dataset"),
    painter=painter,
    config=synth_cfg,
)
dataset_yaml = synth.run()
print("Dataset manifest:", dataset_yaml)
print(Path("dataset/synth_stats.json").read_text())


## 5. Inspect the generated labels — do this before training
Look at the grid: every coloured outline should sit on a painted object, and
every painted object should have an outline. Then compare object sizes with
the real test set (median ~0.2% of the frame). If the keep rate in
`synth_stats.json` is low, or the outlines miss the objects, fix the data
before spending GPU time on training.


In [ ]:
import json

grid = save_label_grid("dataset/images/train", "dataset/preview.jpg", n=24)
plt.figure(figsize=(18, 12)); plt.imshow(Image.open(grid)); plt.axis("off"); plt.show()

print("Synthetic train:", json.dumps(label_size_stats("dataset/images/train"), indent=1))
print("Real test set: ", json.dumps(label_size_stats("test_images", "test_images/labels"), indent=1))


## 6. Train YOLO segmentation
Transfer learning from a COCO-pretrained `yolo11m-seg` checkpoint (auto-downloaded).
Checkpoints, metrics and plots land under the trainer's save dir
(`runs/segment/logs/yolo11m-seg/` with current Ultralytics); the exact path
is printed after training and stored in `weights` below.


In [ ]:
model = train_yolo(dataset_yaml, train_cfg)

# Ultralytics nests runs as runs/<task>/<project>/<name>/, so never
# hardcode the checkpoint path -- ask the trainer where it saved it.
weights = str(Path(model.trainer.save_dir) / "weights" / "best.pt")
print("Best weights:", weights)


## 7. Inference + water contamination index
Run the best checkpoint on an unseen image and compute the
**Surface Plastic Contamination Index** (% of water surface covered by plastic).

In [ ]:
val_images = sorted(Path("dataset/images/val").glob("*.jpg"))
test_path = random.choice(val_images)
print("Test image:", test_path.name)

result = predict(weights, test_path, conf=0.35)
print(f"Detected objects: {len(result.class_ids)}")
for name, score in zip(result.class_names, result.scores):
    print(f"  - {name}: {score:.1%}")
print(f"Contamination index: {contamination_index(result):.2f}% of surface")

visualize(result)


## 8. Evaluate on real photos (`test_images/`)
The training set is synthetic, so the honest test is real photos the model
never saw. The 40 photos are split into **dev (15)** and **test (25)** in
`test_images/SPLITS.csv`: compare runs on `dev`, and only report `test`
when a change is final, so the test number stays a held-out estimate.

`evaluate_dir` writes, under `results/latest/`: per-class P/R/F1 at mask
IoU >= 0.5 (`metrics.csv`), the class-agnostic "plastic" score
(`metrics_agnostic.csv`), per-image counts (`per_image.csv`), the
contamination summary and one visualization per photo. `map_eval` adds
threshold-free mAP50 / mAP50-95, which shows progress even when nothing
clears the confidence threshold.


In [ ]:
import json

headline = evaluate_dir(weights, "test_images", out_dir="results/latest", split="dev",
                        conf=0.35, imgsz=1024)
eval_yaml = make_yolo_eval_dataset("test_images", out_dir="eval_data")
headline.update(map_eval(weights, eval_yaml, split="val", imgsz=1024))  # val = dev photos
print(json.dumps(headline, indent=2))

# When a change is final, score the held-out photos once:
# evaluate_dir(weights, "test_images", out_dir="results/test", split="test")
# map_eval(weights, eval_yaml, split="test")


## 9. What's next
- **Add real training data.** Box-labelled floating-plastic datasets (e.g. Kili
  "Plastic in River", FloW-Img) can be turned into YOLO-seg masks with
  `pws-boxes-to-seg` (box-prompted SAM 2, the same method used for the test
  labels). Train on real + synthetic, or pretrain on synthetic and fine-tune on real.
- Scale up synthesis (`train_count=1000+`) once the §5 preview looks right.
- Small objects: try `imgsz=1280` and tiled inference (SAHI) on large photos.
- Export the best checkpoint to ONNX/TensorRT for edge deployment:
  `yolo export model=<path printed after training> format=onnx`.
- Add more real test photos to `test_images/` (keep `SOURCES.csv` and
  `SPLITS.csv` up to date) and re-run §8 to track progress.
